In [1]:
# transformer-lens 3.4.0 pins torchvision>=0.22,<0.23 (for torch 2.7.x),
# but Colab has torch 2.10 + torchvision 0.25. This breaks pip's resolver.
# Workaround: install both with --no-deps, then install dependencies separately.

# Step 1: Install sae-lens and transformer-lens without the torchvision constraint
!pip install --no-deps transformer-lens sae-lens


# Step 2: Install all actual dependencies (minus torchvision, not needed for text SAEs)
!pip install \
    "transformers>=5.4.0,<6.0.0" \
    "datasets>=3.1.0" \
    "einops>=0.6.0" \
    "jaxtyping>=0.2.11" \
    "fancy-einsum>=0.0.3" \
    "beartype>=0.14.1" \
    "accelerate>=0.23.0" \
    "rich>=12.6.0" \
    "sentencepiece" \
    "typeguard>=4.2,<5" \
    "wandb>=0.13.5" \
    "transformers-stream-generator>=0.0.5,<0.1" \
    "better-abc>=0.0.3" \
    "protobuf>=3.20.0" \
    "huggingface-hub>=0.23.2" \
    "safetensors>=0.4.2,<1.0.0" \
    "plotly>=5.19.0" \
    "nltk>=3.8.1" \
    "python-dotenv>=1.0.1" \
    "pyyaml>=6.0.1" \
    "simple-parsing>=0.1.8" \
    "tenacity>=9.0.0" \
    "babe>=0.0.7"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 31.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 17.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.3/276.3 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.9/236.9 kB 26.7 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=8f33f633793c41df9965342f83add5abf57c603ba9eef2fcb750bbd9b8bb227d
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3

In [3]:
from huggingface_hub import login
import os
from __future__ import annotations

import argparse
import re
import sys
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Iterable

import requests
import torch, gc
from sae_lens import SAE
from transformer_lens import HookedTransformer

from google.colab import drive
drive.mount("/content/drive")

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))
print(torch.cuda.is_bf16_supported())

Mounted at /content/drive
True
0
NVIDIA A100-SXM4-40GB
True


In [4]:
with open("/content/drive/MyDrive/tokens/hftoken.txt") as f:
    token = f.readline().strip()

login(token=token)

In [5]:
DEFAULT_RELEASE = "gemma-scope-2b-pt-res"
DEFAULT_SAE_ID = "layer_20/width_16k/average_l0_71"
DEFAULT_MODEL = "gemma-2-2b"
NEURONPEDIA_FEATURE_API = "https://www.neuronpedia.org/api/feature"

DEVICE = 'cuda'
DTYPE = torch.bfloat16


In [6]:
gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.mem_get_info())

True
NVIDIA A100-SXM4-40GB
(41957130240, 42405855232)


In [7]:
print("----loading SAE----")
sae = SAE.from_pretrained(
    release=DEFAULT_RELEASE,
    sae_id=DEFAULT_SAE_ID,
    device=DEVICE,
    dtype=DTYPE
)

print("----loading hooks----")
metadata = getattr(sae.cfg, "metadata", None)
hook_name = getattr(metadata, "hook_name", None)
match = re.search(r"blocks\.(\d+)\.", hook_name)
hook_layer = int(match.group(1)) if match else None

model_name = getattr(metadata, "model_name", None)
prepend_bos = getattr(metadata, "prepend_bos", None)

print(f"\tHook name:\t{hook_name}")
print(f"\tHook layer:\t{hook_layer}")
print(f"\tModel name:\t{model_name}")
print(f"\tPrepend BOS:\t{prepend_bos}")

print("----loading model----")
model = HookedTransformer.from_pretrained_no_processing(model_name,
                                          device=DEVICE, 
                                          dtype=DTYPE, 
                                          default_prepend_bos=prepend_bos, 
                                          first_n_layers=hook_layer+1)

----loading SAE----


layer_20/width_16k/average_l0_71/params.(…):   0%|          | 0.00/302M [00:00<?, ?B/s]

----loading hooks----
	Hook name:	blocks.20.hook_resid_post
	Hook layer:	20
	Model name:	gemma-2-2b
	Prepend BOS:	True
----loading model----


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model gemma-2-2b into HookedTransformer


In [11]:
@dataclass(frozen=True)
class FeatureHit:
    feature: int
    activation: float
    token_index: int | None = None
    token: str | None = None

@dataclass(frozen=True)
class FeatureInfo:
    description: str | None
    url: str | None
    error: str | None = None

def neuronpedia_source(release: str, sae_id: str) -> tuple[str, str] | None:
    if release != "gemma-scope-2b-pt-res":
        return None
    match = re.search(r"layer_(\d+)/width_(\d+)k/", sae_id)
    if not match:
        return None
    layer, width = match.groups()
    return "gemma-2-2b", f"{layer}-gemmascope-res-{width}k"

def neuronpedia_url(release: str, sae_id: str, feature: int) -> str | None:
    source = neuronpedia_source(release, sae_id)
    if source is None:
        return None
    model_id, source_id = source
    return f"https://www.neuronpedia.org/{model_id}/{source_id}/{feature}"

def neuronpedia_api_url(release: str, sae_id: str, feature: int) -> str | None:
    source = neuronpedia_source(release, sae_id)
    if source is None:
        return None
    model_id, source_id = source
    return f"{NEURONPEDIA_FEATURE_API}/{model_id}/{source_id}/{feature}"

def clean_description(description: str | None) -> str | None:
    if not description:
        return None
    return " ".join(description.split())

def explanation_sort_key(explanation: dict) -> tuple[int, int]:
    type_name = str(explanation.get("typeName", ""))
    model_name = str(explanation.get("explanationModelName", ""))
    scores = explanation.get("scores") or []

    score_bonus = 0
    for score in scores:
        value = score.get("value") if isinstance(score, dict) else None
        if isinstance(value, (int, float)):
            score_bonus = max(score_bonus, int(value))

    preferred_type = 2 if "np_max-act-logits" in type_name else 0
    preferred_model = 1 if "gemini" in model_name.lower() or "gpt-4" in model_name.lower() else 0
    return preferred_type + preferred_model, score_bonus


def best_feature_description(feature_json: dict) -> str | None:
    explanations = feature_json.get("explanations") or []
    if explanations:
        best = max(explanations, key=explanation_sort_key)
        description = clean_description(best.get("description"))
        if description:
            return description

    vector_label = clean_description(feature_json.get("vectorLabel"))
    if vector_label:
        return vector_label

    return None

def fetch_feature_info(release: str, sae_id: str, feature: int, timeout_s: float = 10.0) -> FeatureInfo:
    url = neuronpedia_url(release, sae_id, feature)
    api_url = neuronpedia_api_url(release, sae_id, feature)
    if api_url is None:
        return FeatureInfo(description=None, url=url, error="unsupported Neuronpedia source")

    try:
        response = requests.get(api_url, timeout=timeout_s)
        response.raise_for_status()
        description = best_feature_description(response.json())
        return FeatureInfo(description=description, url=url)
    except Exception as exc:
        return FeatureInfo(description=None, url=url, error=str(exc))

def fetch_feature_infos(
    release: str,
    sae_id: str,
    features: Iterable[int],
    enabled: bool,
    max_workers: int = 8,
) -> dict[int, FeatureInfo]:
    unique_features = sorted(set(features))
    if not enabled:
        return {
            feature: FeatureInfo(description=None, url=neuronpedia_url(release, sae_id, feature))
            for feature in unique_features
        }

    infos: dict[int, FeatureInfo] = {}
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(fetch_feature_info, release, sae_id, feature): feature
            for feature in unique_features
        }
        for future in as_completed(futures):
            feature = futures[future]
            try:
                infos[feature] = future.result()
            except Exception as exc:
                infos[feature] = FeatureInfo(
                    description=None,
                    url=neuronpedia_url(release, sae_id, feature),
                    error=str(exc),
                )
    return infos

def visible_token(token: str, max_len: int = 32) -> str:
    text = repr(token)
    if len(text) <= max_len:
        return text
    return text[: max_len - 4] + "...'"


def top_features_for_vector(
    values: torch.Tensor,
    top_k: int,
    token_index: int | None = None,
    token: str | None = None,
) -> list[FeatureHit]:
    values = values.detach().float().cpu()
    active = torch.nonzero(values > 0, as_tuple=False).flatten()
    if active.numel() == 0:
        return []

    active_values = values[active]
    k = min(top_k, active_values.numel())
    top_values, top_positions = torch.topk(active_values, k=k)
    top_indices = active[top_positions]

    return [
        FeatureHit(
            feature=int(feature.item()),
            activation=float(value.item()),
            token_index=token_index,
            token=token,
        )
        for feature, value in zip(top_indices, top_values, strict=True)
    ]

def top_features_across_prompt(
    feature_acts: torch.Tensor,
    tokens: list[str],
    top_k: int,
) -> list[FeatureHit]:
    acts = feature_acts[0].detach().float().cpu()
    max_values, max_token_indices = acts.max(dim=0)
    hits = top_features_for_vector(max_values, top_k)
    return [
        FeatureHit(
            feature=hit.feature,
            activation=hit.activation,
            token_index=int(max_token_indices[hit.feature].item()),
            token=tokens[int(max_token_indices[hit.feature].item())],
        )
        for hit in hits
    ]

def print_hits(
    title: str,
    hits: Iterable[FeatureHit],
    release: str,
    sae_id: str,
    include_links: bool,
    feature_infos: dict[int, FeatureInfo] | None = None,
    show_descriptions: bool = True,
) -> None:
    print(f"\n{title}")
    hits = list(hits)
    if not hits:
        print("  No active features.")
        return

    for rank, hit in enumerate(hits, start=1):
        token_part = ""
        if hit.token_index is not None and hit.token is not None:
            token_part = f" token={hit.token_index} {visible_token(hit.token)}"
        print(
            f"  {rank:>2}. feature={hit.feature:<7} "
            f"activation={hit.activation:>9.4f}{token_part}"
        )
        info = feature_infos.get(hit.feature) if feature_infos else None
        if include_links:
            url = info.url if info else neuronpedia_url(release, sae_id, hit.feature)
            if url:
                print(f"      link: {url}")
        if not show_descriptions:
            continue
        if info and info.description:
            print(f"      represents: {info.description}")
        elif info and info.error:
            print(f"      represents: unavailable ({info.error})")
        elif feature_infos is not None:
            print("      represents: no Neuronpedia explanation found")


In [14]:
def get_feature_sequence(sae, model, prepend_bos, hook_name, top_k, prompt):
    with torch.inference_mode():
        tokens = model.to_tokens(prompt, prepend_bos=prepend_bos)
        str_tokens = model.to_str_tokens(tokens[0])

        _, cache = model.run_with_cache(tokens, names_filter=[hook_name])
        activations = cache[hook_name]
        feature_acts = sae.encode(activations)

    print("\n" + "=" * 80)
    print(f"Prompt: {prompt!r}")
    print(f"Tokens ({len(str_tokens)}):")
    print("  " + " ".join(f"[{i}] {visible_token(tok)}" for i, tok in enumerate(str_tokens)))

    hits_by_token = [
        top_features_for_vector(feature_acts[0, index], top_k, index, token)
        for index, token in enumerate(str_tokens)
    ]
    #summary_hits = top_features_across_prompt(feature_acts, str_tokens, top_k) if summary else []

    feature_ids = [hit.feature for hits in hits_by_token for hit in hits]
    #feature_ids.extend(hit.feature for hit in summary_hits)

    include_descriptions = True
    if include_descriptions and feature_ids:
        print("\nFetching Neuronpedia explanations...", flush=True)
    feature_infos = fetch_feature_infos(
        DEFAULT_RELEASE,
        DEFAULT_SAE_ID,
        feature_ids,
        enabled=include_descriptions,
    )

    """
    if summary:
        print_hits(
            f"Strongest features anywhere in the prompt (top {top_k})",
            summary_hits,
            sae_info.release,
            sae_info.sae_id,
            include_links,
            feature_infos,
            include_descriptions,
        )
    """

    for index, hits in enumerate(hits_by_token):
        title = f"Token {index} {visible_token(str_tokens[index])} - top {top_k} features"
        print_hits(
            title,
            hits,
            DEFAULT_RELEASE,
            DEFAULT_SAE_ID,
            False,
            feature_infos,
            include_descriptions,
        )
        

In [20]:
get_feature_sequence(sae, model, prepend_bos, hook_name, 5, "USA France London Australia Russia")


Prompt: 'USA France London Australia Russia'
Tokens (6):
  [0] '<bos>' [1] 'USA' [2] ' France' [3] ' London' [4] ' Australia' [5] ' Russia'

Fetching Neuronpedia explanations...

Token 0 '<bos>' - top 5 features
   1. feature=6631    activation=2024.0000 token=0 '<bos>'
      represents: proper nouns.
   2. feature=743     activation= 776.0000 token=0 '<bos>'
      represents: proper nouns and specific terms related to people or entities
   3. feature=5052    activation= 532.0000 token=0 '<bos>'
      represents: the beginning of a document or section, likely signaling the start of significant content
   4. feature=16057   activation= 264.0000 token=0 '<bos>'
      represents: instances of special characters or formatting symbols
   5. feature=9479    activation= 252.0000 token=0 '<bos>'
      represents: the start of a code block or a programming structure

Token 1 'USA' - top 5 features
   1. feature=10194   activation=  89.5000 token=1 'USA'
      represents: references to the Unit

In [ ]:
get_feature_sequence(sae, model, prepend_bos, hook_name, 5, "China Iran Cuba Venezuela Russia")


Prompt: 'China Iran Cuba Venezuela Russia'
Tokens (6):
  [0] '<bos>' [1] 'China' [2] ' Iran' [3] ' Cuba' [4] ' Venezuela' [5] ' Russia'

Fetching Neuronpedia explanations...

Token 0 '<bos>' - top 5 features
   1. feature=6631    activation=2024.0000 token=0 '<bos>'
      represents: proper nouns.
   2. feature=743     activation= 776.0000 token=0 '<bos>'
      represents: proper nouns and specific terms related to people or entities
   3. feature=5052    activation= 532.0000 token=0 '<bos>'
      represents: the beginning of a document or section, likely signaling the start of significant content
   4. feature=16057   activation= 264.0000 token=0 '<bos>'
      represents: instances of special characters or formatting symbols
   5. feature=9479    activation= 252.0000 token=0 '<bos>'
      represents: the start of a code block or a programming structure

Token 1 'China' - top 5 features
   1. feature=13373   activation=  84.5000 token=1 'China'
      represents: country names and thei